In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
Folder_path="/content/drive/MyDrive/Calendar-Assistant NLU"
train_path = os.path.join(Folder_path, "train.json")
val_path   = os.path.join(Folder_path, "val.json")
test_path  = os.path.join(Folder_path, "test.json")

In [3]:
import json
import numpy as np
train_data=json.load(open(train_path))
val_data=json.load(open(val_path))
test_data=json.load(open(test_path))
print(train_data[0])

{'raw_text': 'set a reminder to call emma at midnight', 'tokens': ['set', 'a', 'reminder', 'to', 'call', 'emma', 'at', 'midnight'], 'tags': ['O', 'O', 'O', 'O', 'O', 'B-PERSON', 'O', 'B-TIME'], 'intent': 'SET_REMINDER', 'date_iso': None, 'time_hm': '00:00', 'id': 'ex_002698', 'target_string': 'SET_REMINDER|O O O O O B-PERSON O B-TIME|00:00'}


In [4]:
import torch
!pip install -q gensim
import gensim.downloader as api
fasttext = api.load("fasttext-wiki-news-subwords-300")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 86.0 MB/s eta 0:00:00
[==================================================] 100.0% 958.5/958.4MB downloaded


In [5]:
from collections import Counter

counter = Counter()

for sample in train_data:
    counter.update(sample["tokens"])

word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in counter:
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

print("Vocabulary Size:", len(word2idx))

Vocabulary Size: 649


In [6]:
embedding_dim=300
embedding_matrix=np.random.normal(size=(len(word2idx),embedding_dim))
for word in word2idx:
    if word in fasttext:
        embedding_matrix[word2idx[word]]=fasttext[word]

In [7]:
import torch.nn as nn
embedding=nn.Embedding.from_pretrained(torch.FloatTensor(embedding_matrix),freeze=False)

In [8]:
tag2idx={"<pad>":0}
for sample in train_data:
  for tag in sample["tags"]:
    if tag not in tag2idx :
      tag2idx[tag]=len(tag2idx)
idx2tag={idx:word for word,idx in tag2idx.items()}
print(tag2idx)

{'<pad>': 0, 'O': 1, 'B-PERSON': 2, 'B-TIME': 3, 'B-EVENT': 4, 'I-EVENT': 5, 'B-DATE': 6, 'I-DATE': 7}


In [9]:
X_train = []
Y_train = []
for sample in train_data:
    sentence = [
        word2idx.get(word, word2idx["<UNK>"])
        for word in sample["tokens"]
    ]

    tags = [
        tag2idx[tag]
        for tag in sample["tags"]
    ]

    X_train.append(sentence)
    Y_train.append(tags)
X_test = []
Y_test = []
for sample in test_data:
    sentence = [
        word2idx.get(word, word2idx["<UNK>"])
        for word in sample["tokens"]
    ]

    tags = [
        tag2idx[tag]
        for tag in sample["tags"]
    ]

    X_test.append(sentence)
    Y_test.append(tags)
X_val = []
Y_val= []
for sample in val_data:
    sentence = [
        word2idx.get(word, word2idx["<UNK>"])
        for word in sample["tokens"]
    ]

    tags = [
        tag2idx[tag]
        for tag in sample["tags"]
    ]

    X_val.append(sentence)
    Y_val.append(tags)

In [10]:
maxlen1=max(len(sentence) for sentence in X_train)
maxlen2=max(len(sentence) for sentence in X_test)
maxlen3=max(len(sentence) for sentence in X_val)
maxlen=max(maxlen1,maxlen2,maxlen3)
print(maxlen)

15


In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_train = pad_sequences(
    X_train,
    maxlen=maxlen,
    padding="post",
    value=word2idx["<PAD>"]
)
Y_train = pad_sequences(
    Y_train,
    maxlen=maxlen,
    padding="post",
    value=tag2idx["<pad>"]
)
X_test = pad_sequences(
    X_test,
    maxlen=maxlen,
    padding="post",
    value=word2idx["<PAD>"]
)
Y_test = pad_sequences(
    Y_test,
    maxlen=maxlen,
    padding="post",
    value=tag2idx["<pad>"]
)
X_val = pad_sequences(
    X_val,
    maxlen=maxlen,
    padding="post",
    value=word2idx["<PAD>"]
)
Y_val = pad_sequences(
    Y_val,
    maxlen=maxlen,
    padding="post",
    value=tag2idx["<pad>"]
)

In [13]:
from torch.utils.data import Dataset, DataLoader
class Calendardataset(Dataset):
  def __init__(self,X,Y):
    self.X=X
    self.Y=Y
  def __len__(self):
    return len(self.X)
  def __getitem__(self,idx):
    return torch.LongTensor(self.X[idx]),torch.LongTensor(self.Y[idx])


In [14]:
train_dataset = Calendardataset(X_train, Y_train)
val_dataset = Calendardataset(X_val, Y_val)
test_dataset = Calendardataset(X_test, Y_test)

In [15]:
batch_size=32
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [16]:
for X_batch, Y_batch in train_dataloader:
    print(X_batch.shape)
    print(Y_batch.shape)
    break
for X_batch, Y_batch in val_dataloader:
    print(X_batch.shape)
    print(Y_batch.shape)
    break
for X_batch, Y_batch in test_dataloader:
    print(X_batch.shape)
    print(Y_batch.shape)
    break

torch.Size([32, 15])
torch.Size([32, 15])
torch.Size([32, 15])
torch.Size([32, 15])
torch.Size([32, 15])
torch.Size([32, 15])


In [17]:
class BiLSTMTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, num_tags,embedding_matrix):
        super(BiLSTMTagger, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.FloatTensor(embedding_matrix),freeze=False)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim , num_layers=1, bidirectional=True, batch_first=True)
        self.linear=nn.Linear(hidden_dim*2,num_tags)
    def forward(self, X):
        X = self.embedding(X)
        lstm_out, _ = self.lstm(X)
        output = self.linear(lstm_out)
        return output

In [18]:
embedding_dim=300
hidden_dim=128
num_tags = len(tag2idx)

In [19]:
model = BiLSTMTagger(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    vocab_size=len(word2idx),
    num_tags=num_tags,
    embedding_matrix=embedding_matrix
)

In [20]:
criterion = nn.CrossEntropyLoss(ignore_index=tag2idx["<pad>"])
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)


In [21]:
num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
train_loader = train_dataloader
val_loader = val_dataloader
for epoch in range(num_epochs):

    # =========================
    # TRAINING
    # =========================
    model.train()
    train_loss = 0

    for X_batch, Y_batch in train_loader:

        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        output = model(X_batch)

        loss = criterion(
            output.transpose(1, 2),
            Y_batch
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)


    # =========================
    # VALIDATION
    # =========================
    model.eval()
    val_loss = 0

    with torch.no_grad():

        for X_batch, Y_batch in val_loader:

            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            output = model(X_batch)

            loss = criterion(
                output.transpose(1, 2),
                Y_batch
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)


    # =========================
    # PRINT
    # =========================
    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 1/10 | Train Loss: 0.6772 | Val Loss: 0.0735
Epoch 2/10 | Train Loss: 0.0297 | Val Loss: 0.0278
Epoch 3/10 | Train Loss: 0.0070 | Val Loss: 0.0209
Epoch 4/10 | Train Loss: 0.0034 | Val Loss: 0.0207
Epoch 5/10 | Train Loss: 0.0021 | Val Loss: 0.0201
Epoch 6/10 | Train Loss: 0.0017 | Val Loss: 0.0196
Epoch 7/10 | Train Loss: 0.0010 | Val Loss: 0.0201
Epoch 8/10 | Train Loss: 0.0007 | Val Loss: 0.0198
Epoch 9/10 | Train Loss: 0.0005 | Val Loss: 0.0195
Epoch 10/10 | Train Loss: 0.0004 | Val Loss: 0.0203


In [22]:
!pip install -q seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [23]:
test_loader=test_dataloader
from sklearn.metrics import accuracy_score, classification_report
from seqeval.metrics import f1_score

model.eval()

all_true_tags = []
all_pred_tags = []

with torch.no_grad():

    for X_batch, Y_batch in test_loader:

        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        # Model output:
        # [batch, sequence_length, num_tags]
        output = model(X_batch)

        # Pick the highest-scoring tag for each token
        predictions = output.argmax(dim=2)

        # Move to CPU
        predictions = predictions.cpu().numpy()
        Y_batch = Y_batch.cpu().numpy()

        # Process each sentence
        for true_seq, pred_seq in zip(Y_batch, predictions):

            true_tags = []
            pred_tags = []

            for true_id, pred_id in zip(true_seq, pred_seq):

                # Ignore padding
                if true_id == tag2idx["<pad>"]:
                    continue

                true_tags.append(idx2tag[true_id])
                pred_tags.append(idx2tag[pred_id])

            all_true_tags.append(true_tags)
            all_pred_tags.append(pred_tags)


# ============================================================
# 1. TOKEN ACCURACY
# ============================================================

true_flat = [
    tag
    for sentence in all_true_tags
    for tag in sentence
]

pred_flat = [
    tag
    for sentence in all_pred_tags
    for tag in sentence
]

accuracy = accuracy_score(true_flat, pred_flat)

print(f"Token Accuracy: {accuracy:.4f}")


# ============================================================
# 2. PER-TAG PRECISION / RECALL
# ============================================================

print("\nPer-tag Precision / Recall:")
print(
    classification_report(
        true_flat,
        pred_flat,
        labels=[
            "O",
            "B-DATE", "I-DATE",
            "B-TIME", "I-TIME",
            "B-PERSON", "I-PERSON",
            "B-EVENT", "I-EVENT"
        ],
        zero_division=0
    )
)


# ============================================================
# 3. ENTITY-LEVEL F1
# ============================================================

entity_f1 = f1_score(
    all_true_tags,
    all_pred_tags
)

print(f"Entity-level F1: {entity_f1:.4f}")

Token Accuracy: 0.9941

Per-tag Precision / Recall:
              precision    recall  f1-score   support

           O       0.99      1.00      1.00      3846
      B-DATE       1.00      0.96      0.98       551
      I-DATE       1.00      1.00      1.00       633
      B-TIME       1.00      0.98      0.99       338
      I-TIME       0.00      0.00      0.00         0
    B-PERSON       1.00      1.00      1.00       191
    I-PERSON       0.00      0.00      0.00         0
     B-EVENT       1.00      0.99      1.00       532
     I-EVENT       1.00      0.99      1.00       227

    accuracy                           0.99      6318
   macro avg       0.78      0.77      0.77      6318
weighted avg       0.99      0.99      0.99      6318

Entity-level F1: 0.9862


In [24]:
# ============================================================
# 1. SET SAVE LOCATION
# ============================================================

from google.colab import drive
import os

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/Calendar-Assistant NLU"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Save directory:")
print(SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save directory:
/content/drive/MyDrive/Calendar-Assistant NLU


In [27]:
# ============================================================
# 2. SAVE COMPLETE TASK-2 CHECKPOINT
# ============================================================

task2_checkpoint = {

    # -----------------------------
    # Trained model
    # -----------------------------

    "model_state_dict": model.state_dict(),


    # -----------------------------
    # Word vocabulary
    # -----------------------------

    "word2idx": word2idx,
    "idx2word": idx2word,


    # -----------------------------
    # Embedding
    # -----------------------------

    "embedding_state_dict": embedding.state_dict(),


    # -----------------------------
    # Tag vocabulary
    # -----------------------------

    "tag2idx": tag2idx,
    "idx2tag": idx2tag,


    # -----------------------------
    # Model configuration
    # -----------------------------

    "embedding_dim": 300,
    "hidden_dim": 256,
    "num_tags": num_tags,


    # -----------------------------
    # Training information
    # -----------------------------


}


task2_path = os.path.join(
    SAVE_DIR,
    "task2_final.pt"
)


torch.save(
    task2_checkpoint,
    task2_path
)


print("Task-2 checkpoint saved successfully.")
print("Location:")
print(task2_path)

Task-2 checkpoint saved successfully.
Location:
/content/drive/MyDrive/Calendar-Assistant NLU/task2_final.pt


In [28]:
# ============================================================
# 3. VERIFY TASK-2 CHECKPOINT
# ============================================================

checkpoint = torch.load(
    task2_path,
    map_location="cpu"
)

print("Task-2 checkpoint loaded successfully.\n")

print("Contents:")
print("-" * 40)

for key in checkpoint.keys():
    print(key)

print("-" * 40)

print("\nFile location:")
print(task2_path)

print("\nFile exists:", os.path.exists(task2_path))

Task-2 checkpoint loaded successfully.

Contents:
----------------------------------------
model_state_dict
word2idx
idx2word
embedding_state_dict
tag2idx
idx2tag
embedding_dim
hidden_dim
num_tags
----------------------------------------

File location:
/content/drive/MyDrive/Calendar-Assistant NLU/task2_final.pt

File exists: True
